# Intervener Complexity in Dependency Grammar
## A Cross-Linguistic Computational Study

**CGS410 Course Project**  
Authors: Abhijit Dalai, Kritnandan, Asif Nawaz, Saurabh Kumar, Midhun Manoj

This notebook provides an interactive analysis of the intervener complexity data
produced by the pipeline across 40 languages.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120

## 1. Load Data

In [ ]:
summary_df = pd.read_csv('final_outputs/all_language_summary.csv')
ml_df = pd.read_csv('final_outputs/all_ml_results.csv')
zscore_df = pd.read_csv('final_outputs/all_zscores.csv')

# load language config for typology info
import yaml
with open('config/languages.yaml') as f:
    lang_cfg = yaml.safe_load(f)['languages']

summary_df['typology'] = summary_df['language'].map({k: v.get('typology','') for k,v in lang_cfg.items()})
summary_df['family'] = summary_df['language'].map({k: v.get('family','') for k,v in lang_cfg.items()})

print(f'Languages: {len(summary_df)}')
print(f'ML results: {len(ml_df)} rows')
print(f'Z-scores: {len(zscore_df)} rows')
summary_df.head(10)

## 2. Language Summary Overview

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

metrics = ['avg_complexity', 'avg_dependency_length', 'avg_arity',
           'avg_subtree_size', 'avg_depth', 'avg_efficiency_ratio']
titles = ['Avg Complexity', 'Avg Dependency Length', 'Avg Arity',
          'Avg Subtree Size', 'Avg Depth', 'Avg Efficiency Ratio']

for ax, metric, title in zip(axes.flat, metrics, titles):
    data = summary_df.sort_values(metric, ascending=False)
    colors = data['typology'].map({'SVO':'#4C72B0','SOV':'#DD8452','VSO':'#55A868','Free':'#C44E52'}).fillna('#888888')
    ax.barh(data['language'], data[metric], color=colors)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.invert_yaxis()
    ax.tick_params(axis='y', labelsize=7)

fig.suptitle('Language Summary Metrics (colored by typology)', fontsize=16, fontweight='bold')
fig.tight_layout()
plt.show()

## 3. Typological Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric in zip(axes, ['avg_complexity', 'avg_dependency_length', 'avg_arity']):
    sns.boxplot(data=summary_df, x='typology', y=metric, ax=ax, palette='Set2')
    ax.set_title(metric.replace('avg_','').replace('_',' ').title(), fontsize=13, fontweight='bold')

fig.suptitle('Metrics by Word Order Typology', fontsize=16, fontweight='bold')
fig.tight_layout()
plt.show()

# ANOVA table
anova_df = pd.read_csv('final_outputs/global_stats/typology_anova.csv')
print('ANOVA Results:')
anova_df

## 4. Correlation Analysis

In [ ]:
numeric_cols = summary_df.select_dtypes(include='number').columns
corr = summary_df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix of Language Summary Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Left vs Right Dependencies

In [ ]:
lr_df = pd.read_csv('final_outputs/global_stats/left_right_analysis.csv')
print('Left vs Right Dependency Analysis:')
lr_df.T

## 6. Dravidian Focus

In [ ]:
try:
    drav = pd.read_csv('final_outputs/global_stats/dravidian_analysis.csv')
    print('Dravidian vs Non-Dravidian:')
    display(drav)
except:
    print('Dravidian analysis file not found.')

## 7. Chi-Square Tests

In [ ]:
chi2_per = pd.read_csv('final_outputs/global_stats/chi_square_per_language.csv')
print('Chi-Square per Language (POS Distribution Uniformity):')
display(chi2_per.sort_values('chi2', ascending=False).head(15))

print('\nChi-Square All Metrics Across Languages:')
chi2_metrics = pd.read_csv('final_outputs/global_stats/chi_square_all_metrics.csv')
display(chi2_metrics)

## 8. Z-Score Analysis

In [ ]:
if not zscore_df.empty:
    print('Z-Scores (real vs random baseline):')
    # pivot for display
    pivot = zscore_df.pivot_table(index='language', columns='metric', values='z_score')
    display(pivot)
    
    # plot z-scores
    base_zscores = zscore_df[~zscore_df['metric'].str.contains('grammar')]
    if not base_zscores.empty:
        fig, ax = plt.subplots(figsize=(10, 5))
        for metric in base_zscores['metric'].unique():
            sub = base_zscores[base_zscores['metric'] == metric]
            ax.bar(sub['language'], sub['z_score'], alpha=0.7, label=metric)
        ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
        ax.set_ylabel('Z-Score')
        ax.set_title('Z-Scores: Real vs Random Baseline', fontsize=14, fontweight='bold')
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print('No z-score data available.')

## 9. Machine Learning Results

In [ ]:
if not ml_df.empty and 'language' in ml_df.columns:
    ml_display = ml_df[ml_df['language'].notna()]
    if not ml_display.empty:
        # heatmap
        pivot = ml_display.pivot_table(index='model_name', columns='language', values='f1_score')
        plt.figure(figsize=(12, 5))
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGn',
                    linewidths=0.5, cbar_kws={'label': 'F1 Score'})
        plt.title('ML F1 Scores by Model and Language', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        # bar chart
        avg_f1 = ml_display.groupby('model_name')['f1_score'].mean().sort_values(ascending=False)
        fig, ax = plt.subplots(figsize=(8, 4))
        avg_f1.plot(kind='bar', ax=ax, color='#4C72B0', edgecolor='white')
        ax.set_ylabel('Average F1 Score')
        ax.set_title('Average F1 Score by Model', fontsize=14, fontweight='bold')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No ML results available.')

## 10. KL Divergence Across Typologies

In [ ]:
try:
    kl_df = pd.read_csv('final_outputs/global_stats/typology_kl_divergence.csv')
    print('KL & JS Divergence between Typological Groups:')
    display(kl_df)
except:
    print('KL divergence data not found.')

## 11. PCA — Language Clustering

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

numeric = summary_df.select_dtypes(include='number').fillna(0)
X = StandardScaler().fit_transform(numeric)
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(12, 8))
colors = {'SVO':'#4C72B0','SOV':'#DD8452','VSO':'#55A868','Free':'#C44E52'}

for typ, color in colors.items():
    mask = summary_df['typology'] == typ
    ax.scatter(coords[mask, 0], coords[mask, 1], label=typ, color=color, s=80, alpha=0.8)
    for i, lang in enumerate(summary_df.loc[mask, 'language']):
        idx = summary_df.index[mask].tolist()
        ax.annotate(lang, (coords[idx[list(summary_df.loc[mask,'language']).index(lang)], 0],
                           coords[idx[list(summary_df.loc[mask,'language']).index(lang)], 1]),
                    fontsize=7, alpha=0.7)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=12)
ax.set_title('PCA of Language Complexity Features', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 12. Key Findings Summary

In [ ]:
print('=== KEY FINDINGS ===')
print(f'\n1. Languages analyzed: {len(summary_df)}')
print(f'2. Average complexity score: {summary_df["avg_complexity"].mean():.4f} (std={summary_df["avg_complexity"].std():.4f})')
print(f'3. Average dependency length: {summary_df["avg_dependency_length"].mean():.2f}')
print(f'4. Left dependencies: {summary_df["percent_left_dependencies"].mean():.1f}%')

# typological findings
print('\n5. Complexity by Typology:')
for typ, grp in summary_df.groupby('typology'):
    print(f'   {typ}: mean={grp["avg_complexity"].mean():.4f} (n={len(grp)})')

# ML findings
if not ml_df.empty and 'f1_score' in ml_df.columns:
    print(f'\n6. ML Best Model: {ml_df.groupby("model_name")["f1_score"].mean().idxmax()}')
    print(f'   Best Avg F1: {ml_df.groupby("model_name")["f1_score"].mean().max():.4f}')

print('\n7. ANOVA Results (typology effect):')
if 'metric' in anova_df.columns:
    for _, row in anova_df.iterrows():
        sig = '***' if row['p_value'] < 0.001 else '**' if row['p_value'] < 0.01 else '*' if row['p_value'] < 0.05 else 'ns'
        print(f'   {row["metric"]}: F={row["f_stat"]:.3f}, p={row["p_value"]:.4f} {sig}')

## 13. LLM Generated Sentences vs Real Corpus

To address the secondary question (*"Do LLM-generated sentences exhibit similar intervener distributions as real corpora?"*), we generated sentences using **GPT-2** and **BERT** and parsed them using the same dependency processing pipeline. We then compared the feature distributions of these artificial interveners against the real English corpus.

In [ ]:
from IPython.display import Image, display
import os

llm_plot_dir = "plots/en/llm"
if os.path.exists(llm_plot_dir):
    print("=== LLM Distribution Divergence Heatmap ===")
    display(Image(filename=os.path.join(llm_plot_dir, "llm_divergence_heatmap.png")))
    
    print("\n=== GPT-2 POS Tag Comparison ===")
    display(Image(filename=os.path.join(llm_plot_dir, "llm_gpt2_pos_comparison.png")))
    
    print("\n=== BERT POS Tag Comparison ===")
    display(Image(filename=os.path.join(llm_plot_dir, "llm_bert_pos_comparison.png")))
    
    print("\n=== Complexity Score Comparison (GPT-2 vs Real) ===")
    display(Image(filename=os.path.join(llm_plot_dir, "llm_gpt2_complexity_score_comparison.png")))
else:
    print("LLM comparison plots not found.")